# Visualize Segmentation Results

This notebook loads a trained model from `runs/`, runs inference on samples from `data/splits`,
and visualizes input images, ground-truth masks, predictions, and bounding boxes (if available).

Set `SPLIT` to `train`, `val`, or `test`.
- For `train`/`val`, set `FOLD` (e.g., `fold_0`).
- For `test`, `FOLD` is ignored.


In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib import patches
import numpy as np
import tensorflow as tf
from PIL import Image


2026-01-18 10:42:42.380676: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-18 10:42:42.419057: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-18 10:42:43.416410: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/huay/Projects/PBL4/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:

In [2]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "data" / "splits" / "class_map.txt").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "data" / "splits" / "class_map.txt").exists():
            PROJECT_ROOT = parent
            break

RUNS_DIR = PROJECT_ROOT / "runs"
SPLITS_BASE = PROJECT_ROOT / "data" / "splits"

SPLIT = "val"  # train | val | test
FOLD = "fold_0"  # used for train/val

if SPLIT == "test":
    SPLITS_DIR = SPLITS_BASE
else:
    SPLITS_DIR = SPLITS_BASE / "folds" / FOLD

CLASS_MAP_PATH = SPLITS_BASE / "class_map.txt"

RUN_NAME = "nestnet-resnet18"  # change if you trained a different model
MODEL_PATH = RUNS_DIR / RUN_NAME / "best.keras"
NUM_SAMPLES = 6
SEED = 13
DEFAULT_INPUT_SIZE = (512, 1024)  # (H, W)

BB_MAPS_DIR = SPLITS_DIR / SPLIT / "bb_maps"
SHOW_BB_MAPS = True


In [3]:
def load_class_map(path):
    entries = []
    if not Path(path).exists():
        return entries
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) == 1:
            name, idx = parts[0], int(parts[0])
        else:
            name, idx = parts[0], int(parts[-1])
        entries.append((name, idx))
    return entries


def infer_num_classes(class_map_entries):
    if not class_map_entries:
        return 1
    max_id = max(idx for _, idx in class_map_entries)
    return max_id + 1


def build_colormap(num_classes, seed=13):
    rng = np.random.default_rng(seed)
    colors = rng.uniform(0.0, 1.0, size=(num_classes, 3))
    colors[0] = 0.0
    return colors


def list_pairs(images_dir, masks_dir):
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)
    masks = {p.stem: p for p in masks_dir.glob("*.png")}
    image_paths = []
    for ext in (".jpg", ".jpeg", ".png"):
        image_paths.extend(images_dir.glob(f"*{ext}"))
    pairs = []
    for img in sorted(image_paths):
        mask = masks.get(img.stem)
        if mask is not None:
            pairs.append((img, mask))
    return pairs


def load_image(path, size=None):
    img = Image.open(path).convert("RGB")
    if size is not None:
        img = img.resize((size[1], size[0]), resample=Image.BILINEAR)
    return np.asarray(img).astype(np.float32) / 255.0


def load_mask(path, size=None):
    mask = Image.open(path)
    if size is not None:
        mask = mask.resize((size[1], size[0]), resample=Image.NEAREST)
    arr = np.asarray(mask)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.int32)


def predict_mask(model, image):
    pred = model.predict(image[None, ...], verbose=0)
    if pred.shape[-1] == 1:
        return (pred[0, ..., 0] > 0.5).astype(np.int32)
    return np.argmax(pred[0], axis=-1).astype(np.int32)


def overlay_mask(image, mask, colors, alpha=0.5):
    colored = colors[mask]
    return np.clip((1.0 - alpha) * image + alpha * colored, 0.0, 1.0)


def load_bb_map(path, size=None):
    path = Path(path)
    if not path.exists():
        return None
    data = np.load(path)
    bb_map = data["bb"]
    if size is not None and (bb_map.shape[0], bb_map.shape[1]) != tuple(size):
        resized = []
        for c in range(bb_map.shape[-1]):
            channel = Image.fromarray(bb_map[..., c].astype(np.uint8))
            channel = channel.resize((size[1], size[0]), resample=Image.NEAREST)
            resized.append(np.asarray(channel))
        bb_map = np.stack(resized, axis=-1)
    return bb_map

def bb_map_to_boxes(bb_map):
    boxes = []
    if bb_map is None:
        return boxes
    for class_id in range(1, bb_map.shape[-1]):
        ys, xs = np.where(bb_map[..., class_id] > 0)
        if ys.size == 0:
            continue
        y1, y2 = int(ys.min()), int(ys.max())
        x1, x2 = int(xs.min()), int(xs.max())
        boxes.append((class_id, y1, x1, y2, x2))
    return boxes

def draw_bb_boxes(ax, bb_map, colors, linewidth=2):
    for class_id, y1, x1, y2, x2 in bb_map_to_boxes(bb_map):
        rect = patches.Rectangle(
            (x1, y1),
            max(1, x2 - x1),
            max(1, y2 - y1),
            linewidth=linewidth,
            edgecolor=colors[class_id],
            facecolor='none',
        )
        ax.add_patch(rect)


def infer_num_classes_from_masks(masks_dir):
    masks_dir = Path(masks_dir)
    max_id = 0
    for mask_path in masks_dir.glob("*.png"):
        mask = np.asarray(Image.open(mask_path))
        if mask.ndim == 3:
            mask = mask[..., 0]
        if mask.size:
            max_id = max(max_id, int(mask.max()))
    return max_id + 1 if max_id >= 0 else 1


In [4]:
if not MODEL_PATH.exists():
    raise SystemExit(f"Model not found: {MODEL_PATH}")

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
input_shape = model.input_shape
if isinstance(input_shape, list):
    input_shape = input_shape[0]

if input_shape[1] and input_shape[2]:
    input_size = (int(input_shape[1]), int(input_shape[2]))
else:
    input_size = DEFAULT_INPUT_SIZE

class_map_entries = load_class_map(CLASS_MAP_PATH)
print("Class map:", CLASS_MAP_PATH, "exists=", CLASS_MAP_PATH.exists())
num_classes = infer_num_classes(class_map_entries)
if num_classes <= 1:
    num_classes = infer_num_classes_from_masks(SPLITS_DIR / SPLIT / "masks_semantic")
colors = build_colormap(num_classes)

print("Model:", MODEL_PATH)
print("Input size:", input_size)
print("Classes:", num_classes)


I0000 00:00:1768707763.882025  682038 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6095 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: /home/huay/Projects/PBL4/runs/nestnet-resnet18/best.keras
Input size: (512, 1024)
Classes: 33


In [5]:
pairs = list_pairs(SPLITS_DIR / SPLIT / "img", SPLITS_DIR / SPLIT / "masks_semantic")
if not pairs:
    raise SystemExit(f"No image/mask pairs found in {SPLITS_DIR / SPLIT}")

if SHOW_BB_MAPS and not BB_MAPS_DIR.exists():
    print(f"BB maps not found: {BB_MAPS_DIR}")

rng = np.random.default_rng(SEED)
indices = rng.choice(len(pairs), size=min(NUM_SAMPLES, len(pairs)), replace=False)

fig, axes = plt.subplots(len(indices), 5, figsize=(20, 4 * len(indices)))
if len(indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, idx in enumerate(indices):
    img_path, mask_path = pairs[idx]
    image = load_image(img_path, input_size)
    mask = load_mask(mask_path, input_size)
    pred_mask = predict_mask(model, image)

    max_label = int(max(mask.max(), pred_mask.max()))
    if max_label >= colors.shape[0]:
        colors = build_colormap(max_label + 1)

    bb_map = None
    if SHOW_BB_MAPS:
        bb_path = BB_MAPS_DIR / f"{img_path.stem}.npz"
        bb_map = load_bb_map(bb_path, input_size)

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"Image: {img_path.name}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(colors[mask])
    axes[row, 1].set_title("Ground Truth")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(colors[pred_mask])
    axes[row, 2].set_title("Prediction")
    axes[row, 2].axis("off")

    axes[row, 3].imshow(overlay_mask(image, pred_mask, colors, alpha=0.45))
    axes[row, 3].set_title("Overlay")
    axes[row, 3].axis("off")

    axes[row, 4].imshow(image)
    axes[row, 4].set_title("Bounding Boxes")
    axes[row, 4].axis("off")
    if bb_map is not None:
        draw_bb_boxes(axes[row, 4], bb_map, colors)
    else:
        axes[row, 4].text(0.5, 0.5, "No BB map", ha="center", va="center", color="white")

plt.tight_layout()
plt.show()


SystemExit: No image/mask pairs found in /home/huay/Projects/PBL4/data/splits/folds/fold_0/test

/home/huay/Projects/PBL4/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
